# IMPORTS

In [41]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.svm import SVC, SVR
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, r2_score
from sklearn.ensemble import RandomForestRegressor

# GET DATA

In [42]:
left_features_df = pd.read_csv("dataset/processed/left_hand_features.csv")
# left_features_df = pd.DataFrame(left_features_df)
right_features_df = pd.read_csv("dataset/processed/right_hand_features.csv")
# right_features_df = pd.DataFrame(right_features_df)

In [43]:
left_features_df.head()

,num_taps,mean_peak_amp,std_peak_amp,amp_decrement,mean_iti,std_iti,cv_iti,num_long_pauses,prop_long_pauses,patient_id,hand,updrs,target
0,52,0.895847,0.043548,0.055189,0.542157,0.059807,0.110312,0,0.000000,C34,left,1,0
1,64,0.906656,0.050966,0.060950,0.452910,0.060022,0.132525,1,0.015873,C35,left,0,0
2,69,0.890518,0.125611,0.074415,0.420588,0.068068,0.161841,0,0.000000,C36,left,1,0
3,57,0.807915,0.192839,0.059099,0.522619,0.174785,0.334441,1,0.017857,C37,left,0,0
4,39,0.870205,0.111160,-0.050822,0.660088,0.189742,0.287450,1,0.026316,C38,left,0,0


In [44]:
right_features_df.head()

,num_taps,mean_peak_amp,std_peak_amp,amp_decrement,mean_iti,std_iti,cv_iti,num_long_pauses,prop_long_pauses,patient_id,hand,updrs,target
0,61,0.914892,0.047774,0.075582,0.467500,0.057205,0.122364,0,0.000000,C34,right,0,0
1,73,0.844620,0.148506,-0.074605,0.377778,0.105299,0.278734,4,0.055556,C35,right,0,0
2,60,0.951034,0.019800,0.014383,0.487006,0.054812,0.112550,0,0.000000,C36,right,0,0
3,26,0.934253,0.028889,0.023459,1.145333,0.244746,0.213689,1,0.040000,C37,right,0,0
4,36,0.925534,0.032226,-0.072683,0.778095,0.234648,0.301568,1,0.028571,C38,right,0,0


# MODEL TRAINING

## Left Hand

In [45]:
# Prepare the data
X_left = left_features_df.drop(['patient_id', 'hand', 'updrs', 'target'], axis=1)
y_left = left_features_df['updrs']

# Split the data into training and testing sets
X_train_left, X_test_left, y_train_left, y_test_left = train_test_split(X_left, y_left, test_size=0.2, random_state=42)

# Scale the features
scaler_left = StandardScaler()
X_train_left = scaler_left.fit_transform(X_train_left)
X_test_left = scaler_left.transform(X_test_left)

import joblib
joblib.dump(scaler_left, "models/left_scaler.pkl")


['models/left_scaler.pkl']

### PyTorch

In [46]:
# Convert the data to PyTorch tensors
X_train_left = torch.tensor(np.asarray(X_train_left), dtype=torch.float32)
X_test_left  = torch.tensor(np.asarray(X_test_left), dtype=torch.float32)

y_train_left = torch.tensor(np.asarray(y_train_left), dtype=torch.float32).unsqueeze(1)
y_test_left  = torch.tensor(np.asarray(y_test_left), dtype=torch.float32).unsqueeze(1)

# Create a custom dataset
class UPDRSDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Create data loaders
train_dataset = UPDRSDataset(X_train_left, y_train_left)
test_dataset = UPDRSDataset(X_test_left, y_test_left)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Define the PyTorch model
class UPDRSModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.net(x)

# Initialize the model
left_model = UPDRSModel(X_train_left.shape[1]) 

# Define the loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.AdamW(left_model.parameters(), lr=3e-4, weight_decay=1e-4)

# Train the model
n_epochs = 200

for epoch in range(n_epochs):
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = left_model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    if (epoch+1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{n_epochs}], Loss: {epoch_loss/len(train_loader):.4f}')

# Evaluate the model
left_model.eval()
with torch.no_grad():
    predictions = left_model(X_test_left)
    mse = mean_squared_error(y_test_left.numpy(), predictions.numpy())
    mae = mean_absolute_error(y_test_left.numpy(), predictions.numpy())
    rmse = torch.sqrt(torch.mean((predictions - y_test_left) ** 2))

print(f'Mean Squared Error: {mse:.4f}')
print(f'Mean Absolute Error: {mae:.4f}')
print(f'Root Mean Squared Error: {rmse.item():.4f}')

Epoch [100/200], Loss: 0.1635
Epoch [200/200], Loss: 0.0932
Mean Squared Error: 0.2301
Mean Absolute Error: 0.3840
Root Mean Squared Error: 0.4797


### SVC 

In [47]:
# Train using SVC
left_svc_model = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True)
y_train_left = y_train_left.ravel()
y_test_left = y_test_left.ravel()
left_svc_model.fit(X_train_left, y_train_left)

# Evaluate the model
y_pred_left = left_svc_model.predict(X_test_left)
accuracy = accuracy_score(y_test_left, y_pred_left)
precision = precision_score(y_test_left, y_pred_left)
recall = recall_score(y_test_left, y_pred_left)
f1 = f1_score(y_test_left, y_pred_left)

print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')

joblib.dump(left_svc_model, "models/left_svc.pkl")


Accuracy: 0.7143
Precision: 0.6667
Recall: 0.6667
F1 Score: 0.6667


['models/left_svc.pkl']

### SVR

In [48]:
left_svr_model = SVR(kernel='rbf', C=1.0, gamma='scale')
left_svr_model.fit(X_train_left, y_train_left)

# Evaluate the model
y_pred_left = left_svr_model.predict(X_test_left)

# Round predictions to nearest integer
y_pred_rounded = np.round(y_pred_left)

# Calculate metrics
accuracy = accuracy_score(y_test_left, y_pred_rounded)
precision = precision_score(y_test_left, y_pred_rounded, average='weighted')
recall = recall_score(y_test_left, y_pred_rounded, average='weighted')
f1 = f1_score(y_test_left, y_pred_rounded, average='weighted')

print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')

joblib.dump(left_svr_model, "models/left_svr.pkl")

Accuracy: 0.7143
Precision: 0.7143
Recall: 0.7143
F1 Score: 0.7143


['models/left_svr.pkl']

### Random Forest

In [49]:
left_rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_left, y_train_left)

# Evaluate the model
y_pred_left = model.predict(X_test_left)

# Round predictions to nearest integer
y_pred_rounded = np.round(y_pred)

# Calculate metrics
mse = mean_squared_error(y_test_left, y_pred)
mae = mean_absolute_error(y_test_left, y_pred)
r2 = r2_score(y_test_left, y_pred)
accuracy = accuracy_score(y_test_left, y_pred_rounded)
precision = precision_score(y_test_left, y_pred_rounded, average='weighted')
recall = recall_score(y_test_left, y_pred_rounded, average='weighted')
f1 = f1_score(y_test_left, y_pred_rounded, average='weighted')

print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')
print(f'Mean Squared Error: {mse:.4f}')
print(f'Mean Absolute Error: {mae:.4f}')
print(f'R-squared: {r2:.4f}')

joblib.dump(model, "models/left_rf.pkl")

Accuracy: 0.4286
Precision: 0.1837
Recall: 0.4286
F1 Score: 0.2571
Mean Squared Error: 1.0234
Mean Absolute Error: 0.8835
R-squared: -3.1788


c:\Users\bubut\Documents\SchoolWork\parkinsons-project\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


['models/left_rf.pkl']

## Right Hand

In [50]:
# Prepare the data
X = right_features_df.drop(['patient_id', 'hand', 'updrs', 'target'], axis=1)
y = right_features_df['updrs']

# Split the data into training and testing sets
X_train_right, X_test_right, y_train_right, y_test_right = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the features
scaler_right = StandardScaler()
X_train_right = scaler_right.fit_transform(X_train_right)
X_test_right = scaler_right.transform(X_test_right)

joblib.dump(scaler, "models/right_scaler.pkl")

['models/right_scaler.pkl']

### PyTorch

In [51]:
# Convert the data to PyTorch tensors
X_train_right = torch.tensor(X_train_right, dtype=torch.float32)
X_test_right = torch.tensor(X_test_right, dtype=torch.float32)
y_train_right = torch.tensor(y_train_right.values, dtype=torch.float32).unsqueeze(1) # unsqueeze(1) adds a dimension to the tensor
y_test_right = torch.tensor(y_test_right.values, dtype=torch.float32).unsqueeze(1)

# Create a custom dataset
class UPDRSDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Create data loaders
train_dataset = UPDRSDataset(X_train_right, y_train_right)
test_dataset = UPDRSDataset(X_test_right, y_test_right)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Define the PyTorch model
class UPDRSModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.net(x)

# Initialize the model
right_model = UPDRSModel(X_train_right.shape[1]) 

# Define the loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.AdamW(right_model.parameters(), lr=3e-4, weight_decay=1e-4)

# Train the model
n_epochs = 200

for epoch in range(n_epochs):
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = right_model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    if (epoch+1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{n_epochs}], Loss: {epoch_loss/len(train_loader):.4f}')

# Evaluate the model
right_model.eval()
with torch.no_grad():
    predictions = right_model(X_test_right)
    mse = mean_squared_error(y_test_right.numpy(), predictions.numpy())
    mae = mean_absolute_error(y_test_right.numpy(), predictions.numpy())
    rmse = torch.sqrt(torch.mean((predictions - y_test_right) ** 2))

print(f'Mean Squared Error: {mse:.4f}')
print(f'Mean Absolute Error: {mae:.4f}')
print(f'Root Mean Squared Error: {rmse.item():.4f}')

Epoch [100/200], Loss: 0.2254
Epoch [200/200], Loss: 0.1774
Mean Squared Error: 0.1817
Mean Absolute Error: 0.3957
Root Mean Squared Error: 0.4262


### SVC

In [52]:
# Train using SVC
right_svc_model = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True)
y_train_right = y_train_right.ravel()
y_test_right = y_test_right.ravel()
right_svc_model.fit(X_train_right, y_train_right)

# Evaluate the model
y_pred_right = right_svc_model.predict(X_test_right)
accuracy = accuracy_score(y_test_right, y_pred_right)
precision = precision_score(y_test_right, y_pred_right)
recall = recall_score(y_test_right, y_pred_right)
f1 = f1_score(y_test_right, y_pred_right)

print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')

joblib.dump(right_svc_model, "models/right_svc.pkl")


Accuracy: 0.7143
Precision: 0.7500
Recall: 0.5000
F1 Score: 0.6000


['models/right_svc.pkl']

### SVR

In [53]:
right_svr_model = SVR(kernel='rbf', C=1.0, gamma='scale')
right_svr_model.fit(X_train_right, y_train_right)

# Evaluate the model
y_pred_right = right_svr_model.predict(X_test_right)

# Round predictions to nearest integer
y_pred_rounded_right = np.round(y_pred_right)

# Calculate metrics
accuracy = accuracy_score(y_test_right, y_pred_rounded_right)
precision = precision_score(y_test_right, y_pred_rounded_right, average='weighted')
recall = recall_score(y_test_right, y_pred_rounded_right, average='weighted')
f1 = f1_score(y_test_right, y_pred_rounded_right, average='weighted')

print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')

joblib.dump(right_svr_model, "models/right_svr.pkl")

Accuracy: 0.7143
Precision: 0.7214
Recall: 0.7143
F1 Score: 0.7016


['models/right_svr.pkl']

### Random Forest

In [54]:
right_rf_model = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
right_rf_model.fit(X_train_right, y_train_right)

# Evaluate the model
y_pred_right = right_rf_model.predict(X_test_right)

# Calculate metrics
mse = mean_squared_error(y_test_right, y_pred_right)
mae = mean_absolute_error(y_test_right, y_pred_right)
r2 = r2_score(y_test_right, y_pred_right)
accuracy = accuracy_score(y_test_right, y_pred_rounded_right)
precision = precision_score(y_test_right, y_pred_rounded_right, average='weighted')
recall = recall_score(y_test_right, y_pred_rounded_right, average='weighted')
f1 = f1_score(y_test_right, y_pred_rounded_right, average='weighted')

print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')
print(f'Mean Squared Error: {mse:.4f}')
print(f'Mean Absolute Error: {mae:.4f}')
print(f'R-squared: {r2:.4f}')

joblib.dump(right_rf_model, "models/right_rf.pkl")

Accuracy: 0.7143
Precision: 0.7214
Recall: 0.7143
F1 Score: 0.7016
Mean Squared Error: 0.1538
Mean Absolute Error: 0.3104
R-squared: 0.3721


['models/right_rf.pkl']

In [55]:
# Save models in .pth files

try:
    torch.save(left_model.state_dict(), "models/left_model.pth")
    torch.save(right_model.state_dict(), "models/right_model.pth")
    print("Models saved successfully!")
except Exception as e:
    print(f"Error saving models: {e}")




Models saved successfully!
